# K-Means Clustering

K-Means is an unsupervised algorithm that partitions data into **K clusters** by iteratively assigning points to the nearest centroid and updating centroids.

**Dataset:** Mall Customer Spending Data (Annual Income vs Spending Score)

**Algorithm (Lloyd's):**  
1. Initialise K centroids randomly  
2. Assign each point to the nearest centroid  
3. Recompute each centroid as the mean of its assigned points  
4. Repeat steps 2–3 until convergence (centroids stop moving)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, r"/Jana CMOR/2026_Data_Science_and_Machine_Learning/src/rice_ml/unsupervised_learning")
from rice_ml.unsupervised_learning.kmeans import kmeans
np.random.seed(42)
print("Imports complete")

## Load & Explore the Dataset

We use the **Mall Customers** dataset containing customer spending profiles. Annual income and spending score allow us to segment customers into spending personas.

In [ ]:
import pandas as pd
df = pd.read_csv(r"examples/unsupervised_learning/K-means/Mall_Customers.csv")
print(df.head())
print(f"\nShape: {df.shape}")
print(df.describe())

## Exploratory Data Analysis

Visualising the raw data before clustering reveals natural groupings and confirms K-Means is appropriate.

In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Scatter of raw data
axes[0].scatter(X[:, 0], X[:, 1], color='steelblue', alpha=0.7, edgecolors='k', linewidths=0.4, s=60)
axes[0].set_xlabel("Annual Income (k$)", fontsize=11)
axes[0].set_ylabel("Spending Score", fontsize=11)
axes[0].set_title("Raw Data", fontsize=12, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Histograms
axes[1].hist(X[:, 0], bins=15, color='#457b9d', edgecolor='k', alpha=0.8, label='Annual Income')
axes[1].set_xlabel("Annual Income (k$)", fontsize=11)
axes[1].set_ylabel("Frequency", fontsize=11)
axes[1].set_title("Income Distribution", fontsize=12, fontweight='bold')
axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

axes[2].hist(X[:, 1], bins=15, color='#e63946', edgecolor='k', alpha=0.8, label='Spending Score')
axes[2].set_xlabel("Spending Score", fontsize=11)
axes[2].set_ylabel("Frequency", fontsize=11)
axes[2].set_title("Spending Score Distribution", fontsize=12, fontweight='bold')
axes[2].grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Elbow Method — Choosing K

We compute inertia (within-cluster sum of squares) for K = 1…10. The optimal K lies at the **elbow** — the point where adding another cluster yields diminishing inertia reduction.

In [ ]:
inertias = []
K_range = range(1, 11)
for k in K_range:
    m = kmeans(n_clusters=k, max_iter=300, random_state=42)
    m.fit(X)
    inertias.append(m.inertia_)

# Find the elbow using second derivative
diffs = np.diff(inertias)
diffs2 = np.diff(diffs)
elbow_k = int(np.argmax(diffs2)) + 2  # +2 because double diff shifts index

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Elbow plot
axes[0].plot(K_range, inertias, 'o-', color='steelblue', linewidth=2.5, markersize=8)
axes[0].axvline(elbow_k, color='tomato', linestyle='--', linewidth=2, label=f'Elbow K={elbow_k}')
axes[0].set_xlabel("Number of Clusters (K)", fontsize=12)
axes[0].set_ylabel("Inertia (WCSS)", fontsize=12)
axes[0].set_title("Elbow Method", fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].set_xticks(list(K_range))

# Inertia reduction per step
axes[1].bar(range(2, 11), [-d for d in diffs], color='#2a9d8f', edgecolor='k', alpha=0.8)
axes[1].set_xlabel("K", fontsize=12)
axes[1].set_ylabel("Inertia Reduction", fontsize=12)
axes[1].set_title("Inertia Drop per Additional Cluster", fontsize=13, fontweight='bold')
axes[1].set_xticks(range(2, 11))
axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
print(f"Suggested K (elbow): {elbow_k}")

## Fit K-Means (K=5)

Based on the elbow plot, K=5 captures the five natural spending segments visible in the scatter plot.

In [ ]:
K = 5
model = kmeans(n_clusters=K, max_iter=300, random_state=42)
labels = model.fit_predict(X)

print(f"Converged in {model.n_iter_} iterations")
print(f"Final Inertia: {model.inertia_:,.2f}")
print(f"Cluster sizes: {np.bincount(labels)}")

## Visualise Clusters

Each colour represents a distinct customer segment. Centroids (marked with ★) are the mean position of each cluster.

In [ ]:
COLORS = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#a8dadc']
SEGMENT_NAMES = ['Low Income / Low Spend', 'Low Income / High Spend',
                 'Mid Income / Mid Spend', 'High Income / Low Spend', 'High Income / High Spend']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Cluster scatter with centroids
for k in range(K):
    mask = labels == k
    axes[0].scatter(X[mask, 0], X[mask, 1], color=COLORS[k], label=f'Cluster {k}',
                    alpha=0.75, edgecolors='k', linewidths=0.4, s=60)
axes[0].scatter(model.centroids[:, 0], model.centroids[:, 1],
                marker='*', s=350, color='black', zorder=5, label='Centroids')
axes[0].set_xlabel("Annual Income (k$)", fontsize=12)
axes[0].set_ylabel("Spending Score", fontsize=12)
axes[0].set_title(f"K-Means Clusters (K={K})", fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Cluster centroids bar chart
centroid_income  = model.centroids[:, 0]
centroid_spend   = model.centroids[:, 1]
x_pos = np.arange(K)
width = 0.35
axes[1].bar(x_pos - width/2, centroid_income, width, label='Avg Income', color='#457b9d', alpha=0.8, edgecolor='k')
axes[1].bar(x_pos + width/2, centroid_spend,  width, label='Avg Spend Score', color='#e63946', alpha=0.8, edgecolor='k')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'C{i}' for i in range(K)])
axes[1].set_ylabel("Value", fontsize=12)
axes[1].set_title("Cluster Centroids — Income vs Spending", fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Cluster Profiles

Summarising each cluster's size, mean income, and mean spending score helps interpret the segments as actionable marketing personas.

In [ ]:
print(f"{'Cluster':>9} | {'N':>5} | {'Avg Income':>12} | {'Avg Spending':>13}")
print("-" * 48)
for k in range(K):
    mask = labels == k
    n = np.sum(mask)
    avg_inc = np.mean(X[mask, 0])
    avg_sp  = np.mean(X[mask, 1])
    print(f"{'Cluster ' + str(k):>9} | {n:>5} | {avg_inc:>12.1f} | {avg_sp:>13.1f}")

# Cluster size pie
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sizes = np.bincount(labels)
axes[0].pie(sizes, labels=[f'C{k} (n={s})' for k, s in enumerate(sizes)],
            colors=COLORS, autopct='%1.1f%%', startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title("Cluster Size Distribution", fontsize=12, fontweight='bold')

# Profile radar-style bar chart
x = np.arange(K)
axes[1].scatter(centroid_income, centroid_spend, s=[n * 15 for n in sizes],
                c=COLORS, edgecolors='k', linewidths=1, zorder=3, alpha=0.85)
for k in range(K):
    axes[1].annotate(f'C{k}\n(n={sizes[k]})',
                     (centroid_income[k], centroid_spend[k]),
                     textcoords='offset points', xytext=(8, 5), fontsize=10)
axes[1].set_xlabel("Centroid Annual Income (k$)", fontsize=12)
axes[1].set_ylabel("Centroid Spending Score", fontsize=12)
axes[1].set_title("Centroid Map (bubble size = cluster size)", fontsize=12, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Key Takeaways

- K-Means partitions customers into 5 actionable spending segments.
- The **elbow method** provides a data-driven way to choose K.
- K-Means assumes spherical clusters of roughly equal size — it struggles with non-convex shapes.
- Inertia always decreases with K; the elbow identifies where the improvement plateaus.
- Segment interpretations (Low/Mid/High Income × Low/High Spend) directly inform marketing strategy.